# **HR Attrition EDA**
### **[priyanka lakra]**
---


# **1. LIBRARIES SETUP & ENVIRONMENT**


In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 13
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.titleweight'] = 'bold'
os.makedirs('outputs', exist_ok=True)


In [ ]:

def savefig_and_show(filename):
    plt.savefig(f'outputs/{filename}.png', bbox_inches='tight', dpi=130, transparent=True)
    plt.show()
    plt.clf()


---


# **2. LOAD DATA**


In [ ]:
df = pd.read_csv('hr_attrition_cleaned_data.csv')


In [ ]:
df.columns = df.columns.str.strip().str.lower()
print("Data shape:", df.shape)
print("Sample columns:", df.columns[:8].tolist())
display(df.head(3))


---


# **3. DATA QUALITY CHECKS & VISUALIZATION**


In [ ]:
print("\nMissing Values Per Column:")
print(pd.DataFrame(df.isnull().sum(), columns=['Missing Count']))


In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='mako')
plt.title('Missing Value Heatmap', color='#326798', fontweight='bold')
plt.xlabel('Columns')
plt.ylabel('Records')
plt.tight_layout()
savefig_and_show('missing_value_heatmap')


In [ ]:

print("\nDuplicate Rows:", df.duplicated().sum())


In [ ]:
# Uncomment below to drop duplicates:
# df = df.drop_duplicates()


---


# **4. DATA CLEANING**


In [ ]:
drop_cols = [c for c in ["employeecount", "employeenumber", "over18", "standardhours"] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)


In [ ]:
for col in df.select_dtypes('number').columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes('object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)


---


# **5. TARGET DISTRIBUTION (ATTRITION)**


In [ ]:
print("\nAttrition Value Counts:\n", df['attrition'].value_counts())
print("\nAttrition Crosstab:\n", pd.crosstab(df['attrition'], columns='Count'))
attr_perc = df['attrition'].value_counts(normalize=True) * 100


In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(x=attr_perc.index, y=attr_perc.values, palette=['#2a9d8f', '#e76f51'])
plt.title("Attrition Percentage (%)", color='#e76f51', fontweight="bold")
plt.xlabel("Attrition Status")
plt.ylabel("Percentage (%)")
plt.tight_layout()
savefig_and_show('attrition_percentage_bar')


In [ ]:

plt.figure(figsize=(7,7))
plt.pie(attr_perc, labels=attr_perc.index, autopct='%1.2f%%',
        colors=['#00b4d8', '#f77f00'], startangle=150, textprops={'fontsize':14})
plt.title("Attrition Split", color='#3a3a3a', fontweight='bold')
plt.tight_layout()
savefig_and_show('attrition_pie')


---


# **6. EXPLORATORY DATA ANALYSIS (EDA)**


In [ ]:

# ---- Numeric Features ----
for col in df.select_dtypes('number').columns:
    if col == 'attrition': continue
    print(f"\nCrosstab of {col} Quartiles by Attrition:")
    qcut_col = pd.qcut(df[col], q=4, duplicates='drop')
    print(pd.crosstab(df['attrition'], qcut_col))
    plt.figure(figsize=(8,4))
    sns.histplot(df[col], bins=30, kde=True, color='#2096c7')
    plt.title(f"{col} Value Distribution", fontweight='bold')
    plt.xlabel(f"{col} Value")
    plt.ylabel("Count")
    plt.tight_layout()
    savefig_and_show(f"{col}_hist")


In [ ]:

# ---- Categorical Features ----
for col in df.select_dtypes('object').columns:
    if df[col].nunique() > 25: continue
    print(f"\nCrosstab of {col} by Attrition:")
    print(pd.crosstab(df[col], df['attrition']))
    plt.figure(figsize=(8,4))
    order = df[col].value_counts().index
    sns.countplot(x=col, data=df, order=order, palette='crest')
    plt.title(f"{col} Category Counts", fontweight='bold')
    plt.xlabel(f"{col} Category")
    plt.ylabel("Employee Count")
    plt.xticks(rotation=25)
    plt.tight_layout()
    savefig_and_show(f"{col}_countplot")


In [ ]:

# ---- Numeric Features vs Attrition ----
for col in df.select_dtypes('number').columns:
    if col == 'attrition': continue
    print(f"\nPivot Summary for {col} by Attrition:")
    print(df.pivot_table(index='attrition', values=col, aggfunc=['mean', 'median', 'min', 'max']))
    plt.figure(figsize=(8,5))
    sns.boxplot(x='attrition', y=col, data=df, palette=['#43aa8b', '#f94144'])
    plt.title(f"{col} by Attrition Status", fontweight='bold')
    plt.xlabel("Attrition Status")
    plt.ylabel(f"{col} Value")
    plt.tight_layout()
    savefig_and_show(f"{col}_boxplot")


---


# **7. CORRELATION ANALYSIS & HIGH CORRELATION LIST**


In [ ]:
corr = df.select_dtypes('number').corr()
print("\nCorrelation Matrix (head):")
print(corr.head())


In [ ]:
plt.figure(figsize=(11,9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='turbo', linewidths=0.7, linecolor='grey')
plt.title("Feature Correlation Matrix", fontsize=15, fontweight="bold")
plt.xlabel("Features")
plt.ylabel("Features")
plt.tight_layout()
savefig_and_show('correlation_matrix')


In [ ]:

corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = corr_pairs.stack().reset_index()
high_corr.columns = ["Feature1", "Feature2", "Correlation"]
high_corr = high_corr.loc[high_corr['Correlation'].abs() >= 0.7]
print("\n**Highly Correlated Features (|corr| >= 0.7):**")
print(high_corr if not high_corr.empty else "No highly correlated pairs found.")


---


# **8. OUTLIER REMOVAL & FINAL DATA EXPORT**


In [ ]:
df_clean = df.copy()


In [ ]:
for col in df.select_dtypes('number').columns:
    if col == 'attrition': continue
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    df_clean = df_clean[(df_clean[col] >= lo) & (df_clean[col] <= hi)]
print(f"\nShape after outlier removal: {df_clean.shape}")


In [ ]:

df_clean.to_csv('analyzed_attrition_data.csv', index=False)
print("\nCleaned file saved as analyzed_attrition_data.csv")


---

# **Insights Summary**



### 1. Attrition Level
- Overall attrition rate is **16.12%**, showing a moderately stable workforce.

### 2. Age & Experience Impact
- Employees who left are **younger** (30 vs 35 median age).
- They also have **lower total working experience** (6 vs 9 years).
- They spend **less time at the company** (3 vs 5 years).

### 3. Salary, Job Level & Financial Factors
- Employees who left earn **significantly less** (₹2857 vs ₹4487 monthly income).
- Highest exits are from **Job Level 1**.
- Attrition employees typically have **stock option level 0** (vs 1 for retained).

### 4. Work Environment & Satisfaction
- Lower environment satisfaction among attrition group (2 vs 3).
- No major difference in job satisfaction, involvement, or performance rating.

### 5. Work-life & Overtime Patterns
- Higher attrition among employees with **No Overtime**, hinting disengagement or low workload.
- Work-life balance median is similar for both groups.

### 6. Distance & Travel
- Employees who left live **farther** from office (9 km vs 7 km).
- Highest attrition in **Travel_Rarely** category (0.65 proportion).

### 7. Department & Job Role Risk
- Highest attrition in **Research & Development**.
- By job role, **Laboratory Technicians** show the highest exit rate.

### 8. Demographic Risk Segments
- Higher attrition among **males**.
- **Single** employees leave more than married employees.

### 9. Most Powerful Predictors of Attrition
Top 5 features correlated with attrition:
1. **Years at company** (0.225)
2. **Total working years** (0.218)
3. **Stock option level** (0.208)
4. **Years with current manager** (0.205)
5. **Years in current role** (0.197)
